# 诗歌生成

# 数据处理

In [13]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r', encoding='utf-8') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('poems.txt')
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    # 固定序列长度和 batch 大小，减少 tf.function retracing
    ds = ds.padded_batch(
        100,
        padded_shapes=(tf.TensorShape([200]), tf.TensorShape([])),
        drop_remainder=True
    )
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [11]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    @tf.function
    def call(self, inp_ids):
        '''
        此处完成建模过程，可以参考Learn2Carry
        '''
        inp_emb = self.embed_layer(inp_ids)
        h = self.rnn_layer(inp_emb)
        logits = self.dense(h)
        return logits
    
    @tf.function
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
    
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        h, state = self.rnncell.call(inp_emb, state) # shape(b_sz, h_sz)
        logits = self.dense(h) # shape(b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [14]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [12]:
@tf.function(reduce_retracing=True)
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

@tf.function(reduce_retracing=True)
def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = compute_loss(logits, y, seqlen)

    vars = model.trainable_variables
    grads = tape.gradient(loss, vars)
    grads_and_vars = [(g, v) for g, v in zip(grads, vars) if g is not None]
    if grads_and_vars:
        optimizer.apply_gradients(grads_and_vars)
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [18]:
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

# 先用一个 batch 构建模型参数，避免首次反向传播时无梯度问题
for x0, _, _ in train_ds.take(1):
    _ = model(x0)

for epoch in range(2):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 8.82107
epoch 1 : loss 8.8211155


# 生成过程

In [19]:
def sample_next_id(logits, temperature=0.8, top_k=30):
    # logits shape: [1, vocab_size]
    logits = tf.squeeze(logits, axis=0)
    logits = logits / temperature
    
    if top_k is not None and top_k < logits.shape[-1]:
        values, indices = tf.math.top_k(logits, k=top_k)
        probs = tf.nn.softmax(values)
        chosen = tf.random.categorical(tf.math.log([probs]), 1)[0, 0]
        return int(indices[chosen].numpy())
    
    probs = tf.nn.softmax(logits)
    chosen = tf.random.categorical(tf.math.log([probs]), 1)[0, 0]
    return int(chosen.numpy())

def gen_sentence(begin_word='日', line_len=5, lines=4, temperature=0.8, top_k=30):
    # 五言四句：每句 line_len 个字，共 lines 句
    total_chars = line_len * lines
    state = [tf.zeros(shape=(1, 128), dtype=tf.float32)]

    # 先喂 bos，初始化状态
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    inp_emb = model.embed_layer(cur_token)
    _, state = model.rnncell(inp_emb, state)

    chars = [begin_word]
    cur_token = tf.constant([word2id.get(begin_word, word2id['UNK'])], dtype=tf.int32)

    while len(chars) < total_chars:
        inp_emb = model.embed_layer(cur_token)
        h, state = model.rnncell(inp_emb, state)
        logits = model.dense(h)
        next_id = sample_next_id(logits, temperature=temperature, top_k=top_k)
        token = id2word[next_id]

        # 过滤特殊 token 和常见乱码字符
        if token in ('bos', 'eos', 'PAD', 'UNK', '\u3000', 'ｒ'):
            continue

        chars.append(token)
        cur_token = tf.constant([next_id], dtype=tf.int32)

    out = []
    for i, ch in enumerate(chars, start=1):
        out.append(ch)
        if i % line_len == 0:
            if i < total_chars:
                out.append('，')
            else:
                out.append('。')
    return ''.join(out)

for bw in ['日', '红', '山', '夜', '湖', '海', '月']:
    print(bw + ' -> ' + gen_sentence(bw, line_len=5, lines=4, temperature=0.9, top_k=40))

日 -> 日讷鲕郿跫，暖窗挂艚悸，殁邴铤藜寸，密巫忆固黤。
红 -> 红表含鲵限，笄褕叮溻赞，俘橐浓螯车，图茙隙妩秧。
山 -> 山庾笼清驶，弛裾帜予檇，侵缡布颧芒，姒救燠衙殽。
夜 -> 夜本驔页姑，峻垅翩城妾，蚬但奇槟生，沟创赍笾覉。
湖 -> 湖踏柘孑鉎，粮播薮报濯，瞑艖寐伍籍，跄泣洿蒺铢。
海 -> 海孥匮騄丳，轺瘳凯雕砭，伺剜埽橙艖，怒靡殆缧砢。
月 -> 月矖尤宴装，麈伴曼駊述，繁曆筛柘威，蔗汀綯甪爰。


In [15]:
# 快速验证：仅跑 2 个 step 观察是否出现 retracing 警告
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

for x0, _, _ in train_ds.take(1):
    _ = model(x0)

for i, (x, y, seqlen) in enumerate(train_ds.take(2)):
    loss = train_one_step(model, optimizer, x, y, seqlen)
    print('quick step', i, 'loss', float(loss.numpy()))

quick step 0 loss 8.820348739624023
quick step 1 loss 8.820971488952637
